# Gioia-Thorngren Lattice Chiral Fermion: Formal Verification

**Phase 5a Waves 2+4 — Paper 8: Chirality Wall Master Theorem (Technical Notebook)**

This notebook demonstrates the Gioia-Thorngren (GT) lattice chiral fermion construction
and its formal verification in Lean 4. The GT construction provides the first explicit
3+1D lattice Hamiltonian with exact chiral symmetry [H, Q_A] = 0, verified by
machine-checked proof (Aristotle run `18969de2`).

**Structure:**
1. BdG Hamiltonian and band structure
2. Wilson mass and single Weyl node
3. Chiral charge: non-on-site, non-compact
4. Central theorem: [H, Q_A] = 0
5. GS evasion analysis
6. Three-pillar synthesis

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
from src.core.constants import GT_MODEL, ONSAGER_ALGEBRA, Z16_CLASSIFICATION
from src.core.formulas import gt_wilson_mass, gt_chiral_charge, gt_commutator_identity
from src.chirality.gioia_thorngren import (
    brillouin_zone, wilson_mass, find_weyl_nodes,
    bdg_hamiltonian_fast, band_structure,
    chiral_charge_4x4, chiral_charge_eigenvalues,
    verify_commutator, verify_commutator_tau,
    ginsparg_wilson_check, analyze_gt_evasion,
)
# viz-ref: fig_gt_band_structure, fig_wilson_mass_bz, fig_chiral_charge_spectrum
# viz-ref: fig_gt_commutator_verification, fig_chirality_wall_three_pillars
from src.core.visualizations import (
    COLORS, apply_layout,
    fig_gt_band_structure, fig_wilson_mass_bz,
    fig_chiral_charge_spectrum, fig_gt_commutator_verification,
    fig_chirality_wall_three_pillars,
)

## 1. BdG Hamiltonian and Band Structure

The GT construction uses a modified Karsten-Wilczek Hamiltonian in the
Bogoliubov-de Gennes (Nambu) formalism. At each momentum k, the Hamiltonian
is a 4×4 matrix in σ (spin) ⊗ τ (Nambu) space:

$$H_{\text{BdG}}(\mathbf{p}) = \sin p_1\,\sigma_1\!\otimes\!\mathbf{1}_\tau + \sin p_2\,\sigma_3\!\otimes\!\mathbf{1}_\tau + \sigma_2\!\otimes\! h_{\text{eff}}(\mathbf{p})$$

Lean: `H_BdG`, `h_eff`, `h_tau` in `BdGHamiltonian.lean` (8 theorems, zero sorry)

In [ ]:
# Band structure along high-symmetry path
fig_gt_band_structure().show()

## 2. Wilson Mass and Single Weyl Node

The Wilson mass $M(\mathbf{k}) = 3 - \cos k_x - \cos k_y - \cos k_z$ gaps all
doubler fermions. The key property, proved in `WilsonMass.lean`:

$$M(\mathbf{k}) = 0 \quad\Longleftrightarrow\quad \mathbf{k} = (0,0,0)$$

for all finite lattice sizes $L \geq 1$.

In [ ]:
# Wilson mass heatmap on BZ
fig_wilson_mass_bz().show()

In [ ]:
# Verify single Weyl node on discrete lattices
for L in [4, 8, 12, 16]:
    k = brillouin_zone(L)
    nodes = find_weyl_nodes(k)
    print(f'L={L:3d}: {L**3:5d} k-points, {len(nodes)} Weyl node(s)')

## 3. Chiral Charge: Non-On-Site, Non-Compact

The chiral charge $q_A(\mathbf{p}) = \mathbf{1}_\sigma \otimes [(1+\cos p_3)/2\,\tau_z + (\sin p_3)/2\,\tau_x]$

- **Non-on-site**: depends on $p_3$ (range $R=1$ in real space)
- **Non-compact**: eigenvalues $\pm\cos(p_3/2)$, continuous spectrum
- **Ginsparg-Wilson**: $q_A^2 = \cos^2(p_3/2)\,\mathbf{1}_4$

These properties violate GS conditions I2 (on-site) and I3 (compact).

In [ ]:
# Chiral charge spectrum
fig_chiral_charge_spectrum().show()

In [ ]:
# Ginsparg-Wilson verification
for L in [4, 8, 16]:
    k = brillouin_zone(L)
    gw = ginsparg_wilson_check(k)
    print(f'L={L}: GW max deviation = {gw["max_deviation"]:.1e}, passes: {gw["passes"]}')

## 4. Central Theorem: [H, Q_A] = 0

The heart of the GT construction. Proved in `GTCommutation.lean` (Aristotle run `18969de2`):

$$[H_{\text{BdG}}(\mathbf{k}),\; q_A(\mathbf{k})] = 0 \quad \forall\; \mathbf{k}$$

The proof reduces to a 2×2 τ-space identity via $\sin^2 + \cos^2 = 1$.

In [ ]:
# Numerical verification
fig_gt_commutator_verification().show()

In [ ]:
# Verify for multiple Wilson parameters (the commutator is r-independent)
k = brillouin_zone(8)
for r in [0.1, 0.5, 1.0, 2.0, 5.0]:
    result = verify_commutator(k, r=r)
    print(f'r={r:4.1f}: max|[H,Q_A]| = {result["max_norm"]:.1e}, all zero: {result["all_zero"]}')

## 5. GS Evasion Analysis

The GT model violates exactly the conditions the GS no-go requires.

In [ ]:
report = analyze_gt_evasion(L=8)
print(f'Weyl nodes: {report.weyl_node_count} (chiral: {report.is_chiral})')
print(f'Q_A range: R={report.q_a_range} (on-site: {report.is_on_site})')
print(f'Eigenvalue range: {report.eigenvalue_range} (compact: {report.is_compact})')
print(f'[H, Q_A] = 0 verified: {report.commutator_verified}')
print(f'GS violations: {report.gs_violations}')

## 6. Three-Pillar Synthesis

The chirality wall formal verification comprises three complementary pillars:

| Pillar | Content | Lean Modules | Theorems |
|--------|---------|-------------|----------|
| 1 (No-Go) | GS 9 conditions + TPF evasion | GoltermanShamir, TPFEvasion | 26 |
| 2 (Positive) | GT [H,Q_A]=0 + non-on-site | PauliMatrices through GTWeylDoublet | 56 |
| 3 (Anomaly) | Onsager→su(2)→Witten→Z₁₆ | OnsagerAlgebra through SMGClassification | 103+1ax |

Master synthesis: `ChiralityWallMaster.lean` (17 theorems)

In [ ]:
# Three-pillar overview
fig_chirality_wall_three_pillars().show()

In [ ]:
# Anomaly chain verification
print(f'Onsager DG coefficient: {ONSAGER_ALGEBRA["DG_COEFF"]}')
print(f'su(2) dimension: {ONSAGER_ALGEBRA["SL2_DIM"]}')
print(f'Z16 bordism order: {Z16_CLASSIFICATION["BORDISM_ORDER"]}')
print(f'Witten anomaly: element {Z16_CLASSIFICATION["BORDISM_ORDER"]//2} in Z_{Z16_CLASSIFICATION["BORDISM_ORDER"]}')
print(f'Anomaly cancellation: {Z16_CLASSIFICATION["ANOMALY_CANCELLATION_UNIT"]}n Majorana fermions')